# 11 · Genie Space (natural-language Q&A over the twins)

Creates the **Supply Chain Digital Twins** Genie Space from a `serialized_space` payload built here
(`version: 2`, `data_sources.tables` sorted by identifier, `instructions.text_instructions`). Points
at the 10 twin tables (`*_twin_current` + `*_twin_metrics_timeseries`) with tailored guidance.

Source of truth for the Genie Space — re-run to (re)create.

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("warehouse_id", "7de0b9d9a11160f5", "SQL warehouse id")
dbutils.widgets.text("parent_path", "", "Workspace folder")
C = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
WID = dbutils.widgets.get('warehouse_id').strip()
PARENT = dbutils.widgets.get('parent_path').strip()

## Build serialized_space + create

In [ ]:
import json, uuid
from databricks.sdk import WorkspaceClient

tables = []
for t in ["demand", "inventory", "procurement", "production", "distribution"]:
    tables.append({"identifier": f"{C}.{t}_twin_current"})
    tables.append({"identifier": f"{C}.{t}_twin_metrics_timeseries"})
tables.sort(key=lambda x: x["identifier"])  # API requires sorted identifiers

instruction = (
  "## Role\n"
  "You answer supply-chain questions over the **digital twins** of an agentic supply chain "
  "(Coff-Coff, a coffee roastery + multi-outlet chain), stored in `" + C + "`.\n\n"
  "## Tables\n"
  "* `*_twin_current` = one row per series with the latest state and KPIs (demand: material x plant x channel; "
  "inventory: material x plant; procurement: supplier x material x plant; production: material x plant; "
  "distribution: material x plant x channel).\n"
  "* `*_twin_metrics_timeseries` = per-period history (has `period_start`, `period_type`).\n\n"
  "## Answering\n"
  "* Use `*_twin_current` for now-questions; `*_twin_metrics_timeseries` for trends/history.\n"
  "* Always show `material_text` next to raw `material` codes. Row freshness is in `as_of_date`.\n\n"
  "## Metric conventions\n"
  "* Rates are 0-1 fractions: otif_rate, fill_rate, on_time_rate, in_full_rate, forecast_wmape (render as %).\n"
  "* yield_pct, scrap_pct, first_pass_yield_pct are already 0-100.\n"
  "* Risk flags: is_stockout/is_low_cover/is_overstock, supplier_risk_flag, is_bottleneck, fulfillment_risk_flag.\n"
  "* Numbers 2 decimals, thousands separators for >= 1,000 (not on percentages). Dates YYYY-MM-DD.\n\n"
  "## Data caveats\n"
  "* Cakes (material_group = 'BAKERY_CAKE') are procured, not produced.\n"
  "* Delivery coverage is sparse; demand order lead times are near-zero in the synthetic source, so "
  "backlog is a proxy for ordered-minus-delivered.\n"
)

serialized = {"version": 2, "data_sources": {"tables": tables},
    "instructions": {"text_instructions": [{"id": uuid.uuid4().hex, "content": [instruction]}]}}

w = WorkspaceClient()
body = {"warehouse_id": WID, "serialized_space": json.dumps(serialized),
        "title": "Supply Chain Digital Twins",
        "description": "Natural-language Q&A over the five supply-chain digital twins.",
        "parent_path": PARENT}
resp = w.api_client.do("POST", "/api/2.0/genie/spaces", body=body)
print("space_id:", resp.get("space_id"))
print("Open Genie in the workspace to ask questions, or use the Conversation API.")